In [2]:
pip install transformers torch pandas

In [3]:
Regras_Deterministicas = [
  {
    "id_regra": "R-INT-001",
    "tipo_regra": "INTERACAO",
    "item_A": "IECA",
    "item_B": "POUPADOR_K",
    "gravidade": "CRITICA",
    "alerta": "Risco elevado de Hipercalemia (aumento de potássio no sangue), podendo causar arritmias cardíacas fatais.",
    "fonte": "Diretrizes SBC 2023 / Bulário ANVISA",
    "acao_sugerida": "Monitorar níveis de potássio ou considerar substituição do diurético."
  },
  {
    "id_regra": "R-INT-002",
    "tipo_regra": "INTERACAO",
    "item_A": "IECA",
    "item_B": "AINE",
    "gravidade": "ALTA",
    "alerta": "Risco de Insuficiência Renal Aguda. O AINE antagoniza o efeito vasodilatador renal do IECA.",
    "fonte": "Diretrizes SBC / Nefrologia",
    "acao_sugerida": "Evitar uso prolongado de AINEs. Considerar analgésicos como Paracetamol ou Dipirona."
  },
  {
    "id_regra": "R-INT-003",
    "tipo_regra": "INTERACAO",
    "item_A": "ANTIAGREGANTE",
    "item_B": "AINE",
    "gravidade": "ALTA",
    "alerta": "Aumento significativo do risco de sangramento gastrointestinal. Ibuprofeno pode inibir o efeito cardioprotetor do AAS.",
    "fonte": "Diretrizes SBC / FDA",
    "acao_sugerida": "Se uso concomitante necessário, administrar AAS 2 horas antes do AINE."
  },
  {
    "id_regra": "R-INT-004",
    "tipo_regra": "INTERACAO",
    "item_A": "ESTATINA",
    "item_B": "FIBRATO",
    "gravidade": "ALTA",
    "alerta": "Aumento do risco de miopatia grave e rabdomiólise (destruição muscular crônica).",
    "fonte": "Diretrizes de Dislipidemia SBC",
    "acao_sugerida": "Monitorar enzimas musculares (CPK) e sintomas de dor muscular."
  },
  {
    "id_regra": "R-INT-005",
    "tipo_regra": "INTERACAO",
    "item_A": "NITRATO",
    "item_B": "INIBIDOR_PDE5",
    "gravidade": "CRITICA",
    "alerta": "Risco de hipotensão severa, síncope e infarto do miocárdio. Associação absolutamente contraindicada.",
    "fonte": "Bulário ANVISA / AHA",
    "acao_sugerida": "Uso simultâneo contraindicado. Interromper prescrição imediatamente."
  },
  {
    "id_regra": "R-CON-001",
    "tipo_regra": "CONTRAINDICACAO",
    "item_A": "HIPERTENSAO",
    "item_B": "AINE",
    "gravidade": "MODERADA_A_ALTA",
    "alerta": "AINEs causam retenção de sódio e água, podendo descompensar gravemente a pressão arterial em pacientes hipertensos.",
    "fonte": "Diretrizes Brasileiras de Hipertensão",
    "acao_sugerida": "Preferir analgésicos simples (Paracetamol/Dipirona) ou uso de AINEs em dose mínima pelo menor tempo possível."
  },
  {
    "id_regra": "R-CON-002",
    "tipo_regra": "CONTRAINDICACAO",
    "item_A": "DOENCA_RENAL",
    "item_B": "BIGUANIDA",
    "gravidade": "CRITICA",
    "alerta": "Em pacientes com taxa de filtração glomerular reduzida, o uso de metformina aumenta o risco de acidose lática fatal.",
    "fonte": "Diretrizes SBD 2024",
    "acao_sugerida": "Ajustar dose pela TFG ou suspender se TFG < 30 mL/min/1,73m²."
  },
  {
    "id_regra": "R-CON-003",
    "tipo_regra": "CONTRAINDICACAO",
    "item_A": "ASMA",
    "item_B": "BETABLOQUEADOR_NAO_SELETIVO",
    "gravidade": "CRITICA",
    "alerta": "Risco de broncoespasmo severo e crise asmática grave.",
    "fonte": "Diretrizes SBPT (Pneumologia)",
    "acao_sugerida": "Evitar. Se necessário bloqueio beta, utilizar beta-1 seletivos (ex: Bisoprolol) com cautela."
  },
  {
    "id_regra": "R-CON-004",
    "tipo_regra": "CONTRAINDICACAO",
    "item_A": "ULCERA_PEPTICA",
    "item_B": "AINE",
    "gravidade": "ALTA",
    "alerta": "Risco de exacerbação da úlcera e hemorragia digestiva alta.",
    "fonte": "Federação Brasileira de Gastroenterologia",
    "acao_sugerida": "Preferir inibidores seletivos da COX-2 (Coxibes) associados a IBP se terapia anti-inflamatória for inevitável."
  },
  {
    "id_regra": "R-CON-005",
    "tipo_regra": "CONTRAINDICACAO",
    "item_A": "INSUFICIENCIA_CARDIACA",
    "item_B": "BLOQUEADOR_CANAL_CALCIO_NAO_DIIDROPIRIDINICO",
    "gravidade": "ALTA",
    "alerta": "Fármacos como Verapamil e Diltiazem têm efeito inotrópico negativo, podendo piorar a fração de ejeção e causar descompensação cardíaca.",
    "fonte": "Diretrizes SBC de Insuficiência Cardíaca",
    "acao_sugerida": "Contraindicado em IC com fração de ejeção reduzida. Considerar outras classes anti-hipertensivas."
  }
]

In [4]:
from transformers import pipeline
import torch
import pandas as pd
import requests

print("Iniciando o INTERCEPT CDSS - Motor de Decisão Clínica...\n")

# ==========================================
# 1. O DICIONÁRIO FARMACOLÓGICO (Mapeamento de Classes)
# ==========================================
# Aqui ensinamos ao sistema a qual "família" cada remédio pertence.
Mapeamento_Classes = {
    # IECAs (Inibidores da Enzima Conversora de Angiotensina)
    "captopril": "IECA",
    "enalapril": "IECA",
    "ramipril": "IECA",

    # Poupadores de Potássio
    "espironolactona": "POUPADOR_K",
    "amilorida": "POUPADOR_K",

    # AINEs (Anti-inflamatórios Não Esteroides)
    "ibuprofeno": "AINE",
    "diclofenaco": "AINE",
    "naproxeno": "AINE",

    # Outros exemplos do hackathon
    "metformina": "BIGUANIDA",
    "aas": "ANTIAGREGANTE",
    "aas infantil": "ANTIAGREGANTE"
}

# ==========================================
# 2. O BANCO DE REGRAS CLÍNICAS (A Validação Médica)
# ==========================================
# Baseado na SBC 2023, SBD 2024 e Bulário da ANVISA

df_regras = pd.DataFrame(Regras_Deterministicas)

# ==========================================
# 3. MOTOR DE NLP (Extração)
# ==========================================
device = 0 if torch.cuda.is_available() else -1
print("Carregando BioBERTpt (Aguarde...)...\n")
# O BioBERTpt extrai as entidades médicas do texto livre
ner_pipeline = pipeline("ner", model="pucpr/clinicalnerpt-medical", aggregation_strategy="simple", device=device)

def extrair_e_classificar(texto_prontuario):
    """Extrai o remédio do texto e descobre sua classe terapêutica."""
    entidades = ner_pipeline(texto_prontuario)

    medicamentos_encontrados = []
    for ent in entidades:
        termo = ent['word'].strip().lower()
        # Busca a classe terapêutica no nosso dicionário
        classe = Mapeamento_Classes.get(termo, "DESCONHECIDO")

        # Só adiciona se for um medicamento reconhecido para evitar lixo
        if classe != "DESCONHECIDO":
            medicamentos_encontrados.append({
                "farmaco": termo,
                "classe": classe
            })

    # Remove duplicatas
    return [dict(t) for t in {tuple(d.items()) for d in medicamentos_encontrados}]

# ==========================================
# 4. O MOTOR DE ALERTAS (O "Cérebro" do Sistema)
# ==========================================
def avaliar_prescricao(remedios_uso_continuo, nova_prescricao):
    """Cruza as classes dos remédios do paciente com a nova prescrição."""

    # Descobre a classe do novo remédio prescrito
    classe_nova = Mapeamento_Classes.get(nova_prescricao.lower(), "DESCONHECIDO")

    if classe_nova == "DESCONHECIDO":
        return [{"status": "ERRO", "mensagem": f"Medicamento '{nova_prescricao}' não consta no banco de dados."}]

    alertas_disparados = []

    for med_antigo in remedios_uso_continuo:
        classe_antiga = med_antigo["classe"]

        # Busca na tabela de regras se existe conflito entre a Classe A e Classe B (em qualquer ordem)
        match = df_regras[
            ((df_regras['classe_A'] == classe_antiga) & (df_regras['classe_B'] == classe_nova)) |
            ((df_regras['classe_A'] == classe_nova) & (df_regras['classe_B'] == classe_antiga))
        ]

        if not match.empty:
            regra = match.iloc[0]
            alertas_disparados.append({
                "id_regra": regra['id_regra'],
                "conflito": f"{med_antigo['farmaco'].upper()} ({classe_antiga}) + {nova_prescricao.upper()} ({classe_nova})",
                "gravidade": regra['gravidade'],
                "alerta": regra['alerta'],
                "fonte": regra['fonte']
            })

    return alertas_disparados

# ==========================================
# 5. TESTE DO CASO-CHAVE DO HACKATHON (HIPERCALEMIA)
# ==========================================
print("--- SIMULAÇÃO DE ATENDIMENTO (CASO HIPERCALEMIA) ---")

texto_historico =
"""
S: Pcte 65a, comparece a UBS com queixa de tosse e dores no corpo.
Relata HAS longa data e DM2. Nega alergias conhecidas.
Em uso cont. de enalapriu 20mg 1x/dia, aas e metformina.
O: PA 140x90. FC 82. Corado, hidratado.
A: Mialgia + PA limítrofe.
P: Orientação dieta.
"""

# 1. Extrai o histórico com a IA
perfil_paciente = extrair_e_classificar(texto_historico)
print(f"\n⚙️ IA RECONHECEU: {perfil_paciente}")

# 2. O médico tenta prescrever Espironolactona
remedio_tentativa = "Espironolactona"
print(f"💊 NOVA PRESCRIÇÃO: {remedio_tentativa}")

# 3. O motor avalia as classes
resultados = avaliar_prescricao(perfil_paciente, remedio_tentativa)

print("\n--- RESULTADO DA AVALIAÇÃO ---")
if resultados:
    for alerta in resultados:
        print(f"🚨 INTERCEPT: PRESCRIÇÃO BLOQUEADA (Regra {alerta['id_regra']}) 🚨")
        print(f"⚠️ Fármacos Envolvidos: {alerta['conflito']}")
        print(f"🔴 Gravidade: {alerta['gravidade']}")
        print(f"🩸 Justificativa Clínica: {alerta['alerta']}")
        print(f"📚 Baseado em: {alerta['fonte']}")
else:
    print("✅ Nenhuma interação detectada. Receita liberada.")

Iniciando o INTERCEPT CDSS - Motor de Decisão Clínica...

Carregando BioBERTpt (Aguarde...)...



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: pucpr/clinicalnerpt-medical
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- SIMULAÇÃO DE ATENDIMENTO (CASO HIPERCALEMIA) ---

⚙️ IA RECONHECEU: []
💊 NOVA PRESCRIÇÃO: Espironolactona

--- RESULTADO DA AVALIAÇÃO ---
✅ Nenhuma interação detectada. Receita liberada.
